<center>
<img src="../../img/ods_stickers.jpg" />
    
## [mlcourse.ai](mlcourse.ai) – دورة مفتوحة للتعلم الآلي 
### <center> المؤلف: نيكيتا سيمونوف (لقب ODS Slack: simanjan)</center>
    
## <center> توقع تأخير شركات الطيران</center>



### 1. شرح الميزات والبيانات



يتتبع مكتب إحصاءات النقل (BTS) التابع لوزارة النقل الأمريكية الأداء في الوقت المحدد للرحلات الداخلية التي تديرها شركات النقل الجوي الكبيرة. تظهر معلومات موجزة عن عدد رحلات الطيران في الوقت المحدد والمتأخرة والملغاة والمحولة في تقرير مستهلك السفر الجوي الشهري الذي تصدره وزارة النقل، والذي يتم نشره بعد حوالي 30 يومًا من نهاية الشهر، وكذلك في الجداول الموجزة المنشورة على هذا الموقع. بدأت BTS في جمع التفاصيل حول أسباب تأخير الرحلات الجوية في يونيو 2003. وتم توفير الإحصائيات الموجزة والبيانات الأولية للجمهور في وقت إصدار تقرير مستهلكي السفر الجوي.
تم تجميع هذا الإصدار من مجموعة البيانات من معرض بيانات الرسومات الإحصائية للحوسبة الإحصائية 2009 وهو متاح أيضًا [هنا](http://stat-computing.org/dataexpo/2009/the-data.html). سننظر في بيانات الرحلة لعام 1987.



استيراد كافة المكتبات اللازمة.


In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

import warnings
warnings.filterwarnings('ignore')


تحميل مجموعة البيانات. قم بتغيير المسار إذا لزم الأمر.


In [ ]:
raw_data = pd.read_csv("../../data/1987.csv.bz2")

In [ ]:
raw_data.head()


الأوصاف المتغيرةوصف الاسم    
    - سنة 1987
    - الشهر 1-12
    - يوم الشهر 1-31
    - DayOfWeek 1 (الاثنين) - 7 (الأحد)
    - DepTime وقت المغادرة الفعلي (محلي، همم)
    - CRSDepTime وقت المغادرة المقرر (محلي، همم)
    - وقت الوصول الفعلي لـ ArrTime (محلي، همم)
    - وقت الوصول المقرر CRSArrTime (محلي، همم)
    - رمز الناقل الفريد UniqueCarrier
    - رقم الرحلة رقم الرحلة
    - رقم ذيل الطائرة TailNum
    - الوقت المنقضي الفعلي بالدقائق
    - CRSElapsedTime بالدقائق
    - AirTime في دقائق
    - تأخير وصول ArrDelay، في دقائق
    - DepDelay تأخير المغادرة، في دقائق
    - رمز مطار IATA الأصلي
    - رمز مطار الوجهة IATA
    - المسافة بالأميال
    - سيارة أجرة في سيارة أجرة في الوقت المناسب، في دقائق
    - وقت خروج سيارة الأجرة بالدقائق
    - ألغيت هل تم إلغاء الرحلة؟
    - رمز الإلغاء سبب الإلغاء (A = الناقل، B = الطقس، C = NAS، D = الأمان)
    - منحرف 1 = نعم، 0 = لا
    - تأخير الناقل في دقائق
    - تأخير الطقس في دقائق
    - NASDelay في دقائق
    - تأخير الأمان في دقائق
    - تأخير الطائرات في دقائق



الهدف المستقبلي هو "ملغى".



### 2. تحليل البيانات الأولية


In [ ]:
raw_data.info()

In [ ]:
raw_data.groupby('Cancelled').size()

In [ ]:
pd.crosstab(raw_data['Month'], raw_data['DayOfWeek'])

In [ ]:
raw_data.groupby(['UniqueCarrier','FlightNum'])['Distance'].sum().sort_values(ascending=False).iloc[:5]


عدد جميع الرحلات حسب أيام الأسبوع والأشهر:



### 3. تحليل البيانات المرئية الأولية



مؤامرة الناقل فريدة من نوعها.


In [ ]:
raw_data.groupby('UniqueCarrier').size().plot(kind='bar');


أعلى خمس من أكبر الرحلات الجوية من حيث المسافة الإجمالية.



تحتوي مجموعة البيانات على بيانات لمدة ثلاثة أشهر فقط.
الآن دعونا نلقي نظرة على الرحلات الجوية الملغاة حسب يوم من الأسبوع، ويوم من شهر، وشهر.


In [ ]:
raw_data[raw_data['Cancelled'] == 1].groupby(['DayOfWeek']).size().plot(kind='bar');

In [ ]:
raw_data[raw_data['Cancelled'] == 1].groupby(['DayofMonth']).size().plot(kind='bar');

In [ ]:
raw_data[raw_data['Cancelled'] == 1].groupby(['Month']).size().plot(kind='bar');


الرحلات الملغاة حسب المسافة. هنا تحتاج إلى تطبيع البيانات للحصول على أفضل تمثيل.


In [ ]:
fig, ax = plt.subplots(figsize = (12,6))
ax.hist([raw_data['Distance'], raw_data[raw_data['Cancelled'] == 1]['Distance']], 
        normed=True, label=['All', 'Cancelled'])

ax.set_xlim(0,3000)
ax.set_xlabel('Distance')
ax.set_title('Histogram of Flight Distances')

plt.legend()
plt.show();


الرحلات الجوية الملغاة مؤامرة UniquerCarrier.


In [ ]:
raw_data[raw_data['Cancelled'] == 1].groupby('UniqueCarrier').size().plot(kind='bar');

دعونا نلقي نظرة على أهم خمس رحلات جوية تم إلغاؤها حسب عمودي الأصل والوجهة.


In [ ]:
raw_data[raw_data['Cancelled'] == 1].groupby(['Origin', 'Cancelled']).size()\
.sort_values(ascending=False).iloc[:5].plot(kind='bar');

In [ ]:
raw_data[raw_data['Cancelled'] == 1].groupby(['Dest', 'Cancelled']).size()\
.sort_values(ascending=False).iloc[:5].plot(kind='bar');

In [ ]:
fig, ax = plt.subplots(figsize = (12,6))

ax.hist([raw_data['CRSDepTime'], raw_data[raw_data['Cancelled']== 1]['CRSDepTime']], normed=True,
        label=['All', 'Cancelled'])

ax.set_xlabel('Scheduled Departure Time')
ax.set_title('Histogram of Scheduled Departure Times')

plt.legend()
plt.show()


مؤامرة مربع للمسافة.


In [ ]:
fig, ax = plt.subplots(figsize = (15,6))
sns.boxplot(raw_data['Distance'], ax=ax);


يمكن تغيير حجم عمود المسافة للحصول على نتيجة أفضل.



بناء خريطة الحرارة للارتباطات.


In [ ]:
fig, ax = plt.subplots(figsize=(8,8))
sns.heatmap(raw_data.corr(), ax=ax);


### 4. الرؤى والتبعيات الموجودة



تظهر المخططات أن هناك بيانات مفقودة، بالإضافة إلى بعض العلامات المرتبطة ببعضها البعض.



### 5. اختيار المقاييس



نظرًا لأن لدينا مهمة التصنيف الثنائي 0 أو 1 كمقياس، فيمكننا اختيار:
    - درجة الدقة.
    - أذكر النتيجة.
    - درجة F1.
    - درجة الدقة.



### 6. اختيار النموذج



تحتوي البيانات على ميزات ثنائية وفئوية. لذلك، كنموذج، يمكن اختيار كل من الانحدار المنطقي ونموذج تعزيز التدرج. دعونا نرى كليهما.



دعونا نحاول بناء نموذج.



### 7. المعالجة المسبقة للبيانات



إسقاط الأعمدة التي تحتوي على جميع قيم NaN مثل "TailNum" أو غيرها.


In [ ]:
raw_data.drop(['TailNum', 'AirTime', 'TaxiIn', 'TaxiOut'], axis = 1, inplace=True)


إسقاط جميع أعمدة التأخير وكذلك رمز الإلغاء.


In [ ]:
raw_data.drop(['CancellationCode', 'CarrierDelay',
               'WeatherDelay', 'NASDelay','SecurityDelay', 'LateAircraftDelay'], axis=1, inplace=True)


تحتوي جميع الأعمدة المرتبطة بالرحلات الملغاة على MaN في بعض الأعمدة، مثل "تأخير شركة النقل" أو تأخيرات أخرى. تقديمه بنسبة صفر.


In [ ]:
raw_data.fillna(0, inplace=True)

In [ ]:
raw_data.info()


افصل المتغير المستهدف.


In [ ]:
raw_data_target = raw_data['Cancelled']
raw_data.drop('Cancelled', axis=1, inplace=True)


استبدال شركات النقل الفريدة حسب الفهرس.


In [ ]:
unique_carrier_list = raw_data['UniqueCarrier'].value_counts().index.tolist()

In [ ]:
raw_data['UniqueCarrier'] = raw_data['UniqueCarrier'].apply(lambda x: int(unique_carrier_list.index(x)))


افعل نفس الشيء مع الأصل والوجهة.


In [ ]:
origin_list = raw_data['Origin'].value_counts().index.tolist()

In [ ]:
raw_data['Origin'] = raw_data['Origin'].apply(lambda x : int(origin_list.index(x)))

In [ ]:
dest_list = raw_data['Dest'].value_counts().index.tolist()

In [ ]:
raw_data['Dest'] = raw_data['Dest'].apply(lambda x : int(dest_list.index(x)))


تقسيم مجموعة البيانات والتدريب على الانحدار اللوجستي على مجموعة البيانات.
الآن دعونا نقسم البيانات إلى قسمين.


In [ ]:
from sklearn.model_selection import train_test_split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(raw_data, raw_data_target, test_size=0.33, 
                                                    shuffle=True, random_state=17)


### 8. التحقق من الصحة وتعديل المعلمات الفائقة للنموذج


In [ ]:
from catboost import CatBoostClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

In [ ]:
from sklearn.model_selection import StratifiedKFold

In [ ]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=17)

In [ ]:
%%time
accuracy_score_list = []
recall_score_list = []
f1_score_list = []
precision_score_list = []

for train_index,test_index in skf.split(raw_data,raw_data_target):
    xtr,xvl = raw_data.loc[train_index], raw_data.loc[test_index]
    ytr,yvl = raw_data_target.loc[train_index], raw_data_target.loc[test_index]
    
    log_reg = LogisticRegression()
    log_reg.fit(xtr,ytr)
    predicted = log_reg.predict(xvl)
    
    accuracy_score_list.append(accuracy_score(predicted, yvl))
    recall_score_list.append(recall_score(predicted, yvl))
    f1_score_list.append(f1_score(predicted, yvl))
    precision_score_list.append(accuracy_score(predicted, yvl))

    
print('Accuracy score mean:{}'.format(np.mean(accuracy_score_list)))
print('Recall score mean:{}'.format(np.mean(recall_score_list)))
print('F1 score mean:{}'.format(np.mean(f1_score_list)))
print('Precision score mean:{}'.format(np.mean(precision_score_list)))


بالنظر إلى المقاييس، يمكن للمرء أن يستنتج أن المتغير المستهدف في البيانات به خلل في الفئة.



جرب CatBoostClassifier.


In [ ]:
cat_boost = CatBoostClassifier(random_seed=17, iterations=10)
cat_boost.fit(X_train, y_train, verbose=False, plot=True);

### الجزء 9. إنشاء ميزات جديدة ووصف لهذه العملية



قم بإنشاء ميزة جديدة "DepTimeHour" - وقت المغادرة بالساعة. يمكن أن يكون وقت المغادرة في ساعات الليل سببًا لإلغاء الرحلة. كما تم إنشاء ميزات جديدة مثل "الليل" و"الصباح" و"بعد الظهر" و"المساء".


In [ ]:
raw_data['DepTimeHour'] = raw_data['CRSDepTime'].apply(lambda x: round(x / 100))

raw_data['Night'] = raw_data['DepTimeHour'].apply(lambda x: int(7 >= x >= 0))
raw_data['Morning'] = raw_data['DepTimeHour'].apply(lambda x:  int(12 >= x > 7))
raw_data['Afternoon'] = raw_data['DepTime'].apply(lambda x: int(18 >= x > 12))
raw_data['Evening'] = raw_data['DepTime'].apply(lambda x: int(23 >= x > 18))


### 10. رسم منحنيات التدريب والتحقق من الصحة


In [ ]:
from sklearn.metrics import precision_recall_curve, roc_curve, auc

In [ ]:
precision, recall, thresholds = precision_recall_curve(y_test, cat_boost.predict(X_test))
thresholds_min = np.argmin(np.abs(thresholds))
closest_zero_p = precision[thresholds_min]
closest_zero_r = recall[thresholds_min]

fig, ax= plt.subplots(figsize=(8,8))
ax.plot(precision, recall, label='Precision-Recall Curve')
ax.plot(closest_zero_p, closest_zero_r)
ax.set_xlabel('Precision')
ax.set_ylabel('Recall')
plt.show()


### 11. التنبؤ بعينات الاختبار


In [ ]:
print('Accuracy of LogisticRegression :{}'.format(accuracy_score(log_reg.predict(X_test), y_test)))

In [ ]:
print('Accuracy of Catboost Classifier:{}'.format(accuracy_score(cat_boost.predict(X_test), y_test)))


### 12. الاستنتاجات



نسبة الرحلات الملغاة حسب الإجمالي.


In [ ]:
raw_data_target.value_counts()[1] / raw_data_target.value_counts()[0] * 100


قامت العارضات بعمل رائع في التنبؤ بإلغاء الرحلة. وبعد المعالجة المسبقة للبيانات، أثبتت النماذج أنها جيدة جدًا. وكان السبب في ذلك هو عدم التوازن الطبقي. 
المتغير المستهدف الملغى = 1 هو 1.5 بالمائة فقط من الإجمالي. في نموذج تعزيز التدرج، توقعنا دائمًا المتغير المستهدف بدقة مائة بالمائة، حتى بدون ضبط النموذج.